# Circuit: `mul_8`

Generated from read-only Logisim source files. The port/net graph is canonical; visual trees are derived views.

In [ ]:
from pathlib import Path
import json
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / 'armv4t.circ').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'armv4t.circ').exists(), 'Run from inside the CustomCPU repository'
ATLAS = ROOT / 'ipynb'


In [ ]:
CIRCUIT = 'mul_8'
SLUG = 'mul_8'
arm = json.loads((ATLAS/f'data/armv4t/{SLUG}.json').read_text())
debug = json.loads((ATLAS/f'data/debug/{SLUG}.json').read_text())
delta = json.loads((ATLAS/f'data/diff/{SLUG}.json').read_text())

## Purpose and human audit

In [ ]:
from IPython.display import Markdown, display
paths = sorted((ATLAS/'audits').glob(f'**/{SLUG}*.md'))
if paths:
    for path in paths:
        display(Markdown(f'### {path.relative_to(ATLAS)}\n' + path.read_text()))
else:
    display(Markdown('*Semantic audit pending; machine graph below is complete.*'))

## Interface and component inventory

In [ ]:
def interface_rows(data):
    wanted = set(data['interface']['inputs'] + data['interface']['outputs'])
    rows = []
    for c in data['components']:
        if c['id'] in wanted:
            rows.append({'component': c['id'], 'label': c['label'],
                         'direction': 'input' if c['id'] in data['interface']['inputs'] else 'output',
                         'width': c['attributes'].get('width', '1'), 'facing': c['facing']})
    return pd.DataFrame(rows)
display(interface_rows(arm))
display(pd.DataFrame({'armv4t': arm['inventory'], 'debug': debug['inventory']}).fillna(0).astype(int))

## Every component and port

In [ ]:
component_rows = []
for c in arm['components']:
    if c['ports']:
        for p in c['ports']:
            component_rows.append({'component': c['id'], 'type': c['type'], 'label': c['label'],
                'port': p['name'], 'direction': p['direction'], 'width': p['width'],
                'pin': tuple(p['pin']), 'attributes': c['attributes']})
    else:
        component_rows.append({'component': c['id'], 'type': c['type'], 'label': c['label'],
            'port': None, 'direction': None, 'width': None, 'pin': None, 'attributes': c['attributes']})
components = pd.DataFrame(component_rows)
components

## Every electrical net

In [ ]:
nets = pd.DataFrame(arm['graph']['nets'])
nets[['index', 'name', 'status', 'drivers', 'ports', 'labels']]

## Every directed signal connection

In [ ]:
connections = pd.DataFrame(arm['graph']['edges'])
connections['instruction'] = connections['src'] + '.wire -> ' + connections['dst']
connections[['net', 'instruction', 'src', 'dst']]

## State, feedback, and condensation tree

In [ ]:
signal = nx.read_graphml(ATLAS/f'data/armv4t/{SLUG}.signal.graphml')
scc = sorted(nx.strongly_connected_components(signal), key=len, reverse=True)
pd.DataFrame([{'size': len(group), 'members': sorted(group)} for group in scc if len(group) > 1])

## Component-level graph

In [ ]:
cg = nx.read_graphml(ATLAS/f'data/armv4t/{SLUG}.component.graphml')
plt.figure(figsize=(18, 12))
pos = {node: (float(data.get('x', 0)), -float(data.get('y', 0))) for node, data in cg.nodes(data=True)}
nx.draw_networkx_nodes(cg, pos, node_size=300, alpha=.75)
nx.draw_networkx_edges(cg, pos, arrows=True, width=.6, alpha=.45)
if len(cg) <= 80:
    nx.draw_networkx_labels(cg, pos, font_size=5)
plt.axis('off'); plt.show()

## Health and model coverage

In [ ]:
display(pd.DataFrame([{'file': 'armv4t', **arm['coverage'], **arm['health']},
                      {'file': 'debug', **debug['coverage'], **debug['health']}]))

## `armv4t` → `debug` delta

In [ ]:
display(Markdown('**' + delta['summary'] + '**'))
display(pd.DataFrame({
 'only_arm_node': pd.Series(delta['only_a_nodes']),
 'only_debug_node': pd.Series(delta['only_b_nodes'])}))
display(pd.DataFrame({
 'only_arm_connection': pd.Series([' -> '.join(x) for x in delta['only_a_connections']]),
 'only_debug_connection': pd.Series([' -> '.join(x) for x in delta['only_b_connections']])}))